# experiment_beijing

## Reproducibility bootstrap
Run first. Resolves paths for the authors' Drive, a fresh Colab (clones the anon repo), or a local clone. No edits needed.

In [ ]:
# === Reproducibility bootstrap (public bundle) ===
# Resolves all paths for: (a) authors' Google Drive, (b) fresh Colab (clones repo),
# (c) local clone. Sets CODE_DIR, DATA_DIR, RESULTS_DIR, DATA_PATH. Run first; no edits needed.
import os, sys
from pathlib import Path

RESULTS_SUBFOLDER = "experiment_beijing"
DATA_FILENAME = "beijing_multisite.csv"   # None for synthetic experiments

def _resolve():
    try:
        import google.colab  # noqa: F401
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
        dr = Path('/content/drive/MyDrive')
        if (dr/'GNAVAR'/'code'/'gnavar_core.py').exists():
            b = dr/'GNAVAR'; return b/'code', b/'data', b/'results'
        # Fresh Colab without the authors' Drive: the repo files must be present in the
        # session. Anonymous-review repos cannot be git-cloned, so upload the bundle:
        #   1) Download the ZIP from the Anonymous GitHub page (Download / ZIP button).
        #   2) In Colab, upload the ZIP via the Files pane, then in a cell run:
        #        !unzip -o your_bundle.zip
        #   3) %cd into the unzipped repo folder, then run this notebook.
        for cand in [Path('/content')/'ICDM-GNAVAR-EDAE', Path.cwd()]:
            if (cand/'src'/'gnavar_core.py').exists():
                return cand/'src', cand/'data', cand/'results'
        raise FileNotFoundError(
            'Repo files not found in the Colab session. Download the ZIP from the '
            'Anonymous GitHub page, upload and unzip it here, then %cd into the folder '
            'and re-run. See the repository README, Path B, Option B1.')
    except ImportError:
        repo = Path.cwd()
        while repo != repo.parent and not (repo/'verify_paper_numbers.py').exists():
            repo = repo.parent
        return repo/'src', repo/'data', repo/'results'

CODE_DIR, DATA_DIR, RESULTS_ROOT = _resolve()
sys.path.insert(0, str(CODE_DIR))
RESULTS_DIR = RESULTS_ROOT / RESULTS_SUBFOLDER
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
DATA_PATH = (DATA_DIR / DATA_FILENAME) if DATA_FILENAME else None
DRIVE_ROOT = str(CODE_DIR.parent.parent)  # back-compat for any cell referencing DRIVE_ROOT
print('CODE_DIR    =', CODE_DIR)
print('RESULTS_DIR =', RESULTS_DIR)
if DATA_PATH: print('DATA_PATH   =', DATA_PATH)
import numpy as np
import pandas as pd
import json, hashlib, datetime, time, itertools, platform
try:
    import torch
    import torch.nn as nn
except Exception:
    pass
from gnavar_core import *  # model, generator, fit/eval utils
# === end bootstrap ===


# Beijing Air Quality: Real-Data G-NAVAR Application

**Purpose.** Test G-NAVAR's diagnostic and recovery claims on real atmospheric chemistry data, the Beijing Multi-Site Air Quality dataset.

## Hypotheses

**Setup A: O3 photochemistry.** Target: ozone (O3). Sources: NO2, TEMP, WSPM, PRES, RAIN.
  - **Expected modulator**: TEMP, on the NO2 → O3 edge. Textbook photochemistry says O3 formation from NO2 photolysis accelerates with temperature.

**Setup B: PM10 dispersion.** Target: PM10. Sources: PM2.5, WSPM, TEMP, DEWP.
  - **Expected modulator**: WSPM, on the PM2.5 → PM10 edge. Wind speed modulates the resuspension and dispersion of coarse particulate matter.

## Caveats on the 'expected' interactions

Both interactions are textbook chemistry, but Beijing's heavily anthropogenic pollution profile may modify the relationship. The 'expected modulator' is a hypothesis, not ground truth.
If G-NAVAR doesn't find it, that could mean (a) the model failed, (b) the data doesn't support the mechanism at hourly resolution, or (c) a confound (e.g., NO2 and TEMP anti-correlated in Beijing winter).

## Design

- **4 sites** (most usable rows + geographic diversity): Nongzhanguan, Tiantan, Huairou, Gucheng
- **Per-site independent fits** (no pooling). 4 sites × 2 setups = 8 (site, setup) pairs.
- **Run-aware lag tensor**: lag pairs computed within each contiguous run only.
- **Chronological 80/20 train/test split** on the run sequence (first 80% of runs train, last 20% test).
- **Data is already z-scored per site per variable** in the input file (verified at load time).
- **G-NAVAR with restart-and-keep-best** (n_restarts=3); same for Pairwise NAVAR.

## Reporting per (site, setup)

1. Effective rank of the joint lag-block covariance (pre-fit diagnostic)
2. G-NAVAR vs Pairwise NAVAR held-out forecast MSE
3. Modulator set discovered by G-NAVAR for each edge
4. 'Hypothesis check': did G-NAVAR find the expected modulator?
5. Cross-site consistency: across 4 sites, how often is the expected modulator recovered?

## Outputs (under `/content/drive/MyDrive/GNAVAR/results/experiment_beijing/`)

- `results.csv`: per-(site, setup) metrics
- `modulators.csv`: full modulator-set tables per fit
- `summary.txt`: cross-site consistency report and hypothesis verdict
- `metadata.json`: reproducibility metadata

## Cell 1: Drive mount and data file

## Cell 2: Imports

In [ ]:
from gnavar_core import *
# Brings in:
#   Config, GNAVAR, PairwiseNAVAR
#   make_lag_tensor_runs (run-aware), effective_rank_full
#   fit_gnavar_from_lag, fit_gnavar_from_lag_with_restarts
#   fit_pairwise_from_lag, fit_pairwise_from_lag_with_restarts
#   held_out_mse_gnavar_from_lag, held_out_mse_pairwise_from_lag
#   detect_all_modulators, gate_triviality_score
#   DEVICE, USE_AMP

import time, json, hashlib, platform, datetime
from dataclasses import asdict
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

print(f'Device: {DEVICE} | Mixed precision: {USE_AMP}')
if torch.cuda.is_available():
    print(f'CUDA device: {torch.cuda.get_device_name(0)}')

## Cell 3: Setups and config

In [ ]:
# Variable order in the input file (from manifest)
NODES = ['PM2.5', 'PM10', 'SO2', 'NO2', 'CO', 'O3', 'TEMP', 'PRES', 'DEWP', 'RAIN', 'WSPM']

# Setup A: O3 photochemistry
#   Target: O3.  Sources: NO2, TEMP, WSPM, PRES, RAIN
#   Hypothesis: TEMP modulates the NO2 -> O3 edge.
SETUP_A = {
    'name': 'A_O3_photochem',
    'target': 'O3',
    'sources': ['NO2', 'TEMP', 'WSPM', 'PRES', 'RAIN'],
    'expected_edge_source': 'NO2',
    'expected_modulator':   'TEMP',
    'description': 'NO2 -> O3 edge expected to be modulated by TEMP (photochemistry)',
}

# Setup B: PM10 dispersion
#   Target: PM10. Sources: PM2.5, WSPM, TEMP, DEWP
#   Hypothesis: WSPM modulates the PM2.5 -> PM10 edge.
SETUP_B = {
    'name': 'B_PM10_dispersion',
    'target': 'PM10',
    'sources': ['PM2.5', 'WSPM', 'TEMP', 'DEWP'],
    'expected_edge_source': 'PM2.5',
    'expected_modulator':   'WSPM',
    'description': 'PM2.5 -> PM10 edge expected to be modulated by WSPM (dispersion)',
}

SETUPS = [SETUP_A, SETUP_B]
SITES = ['Nongzhanguan', 'Tiantan', 'Huairou', 'Gucheng']

# Training config -- same training conventions as synthetic experiments
K = 2
BASE_SEED = 42
N_RESTARTS = 3
TEST_RUN_FRACTION = 0.20

# n_vars is set per setup
def cfg_for_setup(setup):
    n_vars = 1 + len(setup['sources'])
    return Config(
        n_vars=n_vars,
        K=K,
        hidden_dim=32,
        n_epochs=300,
        l1_lambda=0.005,
        triviality_threshold=0.001,
    )

print(f'Sites: {SITES}')
print(f'Setup A: target={SETUP_A["target"]}, sources={SETUP_A["sources"]}')
print(f'Setup B: target={SETUP_B["target"]}, sources={SETUP_B["sources"]}')
print(f'\nTotal (site, setup) pairs: {len(SITES) * len(SETUPS)}')
print(f'Total fits: {len(SITES) * len(SETUPS) * 2 * N_RESTARTS}  (2 models x 3 restarts each)')

## Cell 4: Load Beijing data and verify normalization

In [ ]:
data = np.load(DATA_PATH)
print(f'Loaded {DATA_PATH.name}: {len(data.files)} arrays')

# Verify each chosen site's data is in the file and check normalization
print(f'\nPer-site verification (z-score sanity check):')
print(f'{"site":<15s}{"n_rows":>8s}{"n_runs":>8s}{"usable":>8s}{"mean range":>16s}{"std range":>14s}')
for site in SITES:
    X = data[f'X__{site}']
    runs = data[f'runs__{site}']
    # Build mask of usable rows
    mask = np.zeros(X.shape[0], dtype=bool)
    for s, e in runs:
        mask[s:e] = True
    X_usable = X[mask]
    mean_range = f'[{X_usable.mean(axis=0).min():.2e}, {X_usable.mean(axis=0).max():.2e}]'
    std_range = f'[{X_usable.std(axis=0).min():.3f}, {X_usable.std(axis=0).max():.3f}]'
    print(f'{site:<15s}{X.shape[0]:>8d}{len(runs):>8d}{mask.sum():>8d}  {mean_range}  {std_range}')

print('\nIf means are ~1e-6 and stds are ~1.0, data is already z-scored. Good.')

# Variable indices for the two setups
def setup_indices(setup):
    """Returns indices [target_idx, source1_idx, source2_idx, ...] into the NODES list."""
    idx = [NODES.index(setup['target'])]
    idx.extend(NODES.index(v) for v in setup['sources'])
    return idx

for s in SETUPS:
    print(f"  Setup {s['name']}: cols={setup_indices(s)}")

## Cell 5: Trial driver (per site, per setup)

In [ ]:
def run_beijing_trial(site: str, setup: dict, data, base_seed: int = BASE_SEED) -> dict:
    """
    For a single (site, setup):
      1. Slice the relevant columns from the site's data matrix.
      2. Build run-aware lag tensors with chronological train/test split.
      3. Compute effective rank (pre-fit diagnostic).
      4. Fit G-NAVAR with restarts and Pairwise with restarts.
      5. Report MSE comparison and modulator sets.
    """
    t0 = time.time()
    X_full = data[f'X__{site}']
    runs   = data[f'runs__{site}']
    cols   = setup_indices(setup)
    X_setup = X_full[:, cols]  # target at column 0

    # Chronological 80/20 split on RUNS (not on hours)
    n_runs = len(runs)
    split_idx = int(n_runs * (1.0 - TEST_RUN_FRACTION))
    train_runs = runs[:split_idx]
    test_runs  = runs[split_idx:]

    X_lag_train, y_train = make_lag_tensor_runs(X_setup, train_runs, K=K, target_col=0)
    X_lag_test,  y_test  = make_lag_tensor_runs(X_setup, test_runs,  K=K, target_col=0)
    if X_lag_train.shape[0] == 0 or X_lag_test.shape[0] == 0:
        raise ValueError(f'Empty lag tensor for {site}/{setup["name"]}: '
                         f'train={X_lag_train.shape[0]}, test={X_lag_test.shape[0]}')

    # Pre-fit diagnostic
    r_eff = effective_rank_full(X_lag_train)

    cfg = cfg_for_setup(setup)

    print(f'  [setup={setup["name"]}, site={site}]', flush=True)
    print(f'    train samples: {X_lag_train.shape[0]}, test: {X_lag_test.shape[0]}, '
          f'r_eff: {r_eff:.3f}', flush=True)

    # G-NAVAR
    t1 = time.time()
    m_gn = fit_gnavar_from_lag_with_restarts(X_lag_train, y_train, cfg,
                                              seed=base_seed + 1000,
                                              n_restarts=N_RESTARTS, verbose=False)
    t_gn = time.time() - t1
    mse_gn_train = held_out_mse_gnavar_from_lag(m_gn, X_lag_train, y_train)
    mse_gn_test  = held_out_mse_gnavar_from_lag(m_gn, X_lag_test, y_test)

    # Pairwise NAVAR
    t1 = time.time()
    m_pw = fit_pairwise_from_lag_with_restarts(X_lag_train, y_train, cfg,
                                                seed=base_seed + 1000,
                                                n_restarts=N_RESTARTS, verbose=False)
    t_pw = time.time() - t1
    mse_pw_train = held_out_mse_pairwise_from_lag(m_pw, X_lag_train, y_train)
    mse_pw_test  = held_out_mse_pairwise_from_lag(m_pw, X_lag_test, y_test)

    # Modulator sets (G-NAVAR)
    X_lag_t = torch.from_numpy(X_lag_train).to(DEVICE)
    mods = detect_all_modulators(m_gn, X_lag_t, threshold=cfg.triviality_threshold)
    src_names = setup['sources']
    mod_dict = {src_names[j]: sorted(src_names[k] for k in modset)
                for j, modset in mods.items()}

    # Per-edge triviality scores (numerical detail behind the binary modulator set)
    triv_scores = {}
    for j in range(len(src_names)):
        for k in range(len(src_names)):
            if k == j: continue
            score = gate_triviality_score(m_gn, X_lag_t, j, k)
            triv_scores[f'{src_names[j]}<-{src_names[k]}'] = score

    # Hypothesis check: was the expected modulator found?
    exp_src = setup['expected_edge_source']
    exp_mod = setup['expected_modulator']
    hypothesis_modulator_found = exp_mod in mod_dict.get(exp_src, [])
    # Also report the triv score for the expected edge specifically
    j_exp = src_names.index(exp_src)
    k_exp = src_names.index(exp_mod)
    exp_edge_triv = gate_triviality_score(m_gn, X_lag_t, j_exp, k_exp)

    return {
        'site': site,
        'setup': setup['name'],
        'target': setup['target'],
        'n_train': X_lag_train.shape[0],
        'n_test':  X_lag_test.shape[0],
        'r_eff': r_eff,
        'mse_train_gnavar': mse_gn_train,
        'mse_train_pairwise': mse_pw_train,
        'mse_test_gnavar': mse_gn_test,
        'mse_test_pairwise': mse_pw_test,
        'mse_ratio_pw_to_gn': mse_pw_test / mse_gn_test if mse_gn_test > 0 else float('inf'),
        'expected_edge': f'{exp_src}->{setup["target"]} modulated by {exp_mod}',
        'hypothesis_modulator_found': hypothesis_modulator_found,
        'expected_edge_triv_score': exp_edge_triv,
        'modulator_dict_json': json.dumps(mod_dict),
        'triv_scores_json': json.dumps(triv_scores),
        'gnavar_fit_seconds': t_gn,
        'pairwise_fit_seconds': t_pw,
        'total_elapsed_seconds': time.time() - t0,
    }

## Cell 6: Resume-aware sweep driver

In [ ]:
RESULTS_CSV = RESULTS_DIR / 'results.csv'

def load_existing():
    if RESULTS_CSV.exists():
        df = pd.read_csv(RESULTS_CSV)
        return df, set(zip(df['site'], df['setup']))
    return pd.DataFrame(), set()

def append_result(row):
    write_header = not RESULTS_CSV.exists()
    pd.DataFrame([row]).to_csv(RESULTS_CSV, mode='a', header=write_header, index=False)

def run_beijing_sweep():
    _, completed = load_existing()
    print(f'Already completed: {len(completed)} (site, setup) pairs')
    plan = [(site, setup) for setup in SETUPS for site in SITES]
    todo = [(s, su) for (s, su) in plan if (s, su['name']) not in completed]
    print(f'To run: {len(todo)} (site, setup) pairs')
    for i, (site, setup) in enumerate(todo, 1):
        print(f'\n[{i}/{len(todo)}] site={site}, setup={setup["name"]}', flush=True)
        result = run_beijing_trial(site=site, setup=setup, data=data)
        append_result(result)
        print(f'  {result["total_elapsed_seconds"]:.1f}s | '
              f'r_eff={result["r_eff"]:.3f} | '
              f'MSE Gn={result["mse_test_gnavar"]:.4f} vs Pw={result["mse_test_pairwise"]:.4f} '
              f'(ratio {result["mse_ratio_pw_to_gn"]:.2f}x) | '
              f'expected modulator found: {result["hypothesis_modulator_found"]}', flush=True)
    return load_existing()[0]

## Cell 7: Reproducibility metadata

In [ ]:
def _sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(8192), b''):
            h.update(chunk)
    return h.hexdigest()

import gnavar_core as _gc
md = {
    'timestamp': datetime.datetime.utcnow().isoformat() + 'Z',
    'experiment': 'beijing_real_data',
    'data_file_sha256': _sha256(DATA_PATH),
    'gnavar_core_path': _gc.__file__,
    'gnavar_core_sha256': _sha256(_gc.__file__),
    'sites': SITES,
    'setups': [{'name': s['name'], 'target': s['target'], 'sources': s['sources'],
                'expected_edge_source': s['expected_edge_source'],
                'expected_modulator': s['expected_modulator']} for s in SETUPS],
    'K': K,
    'n_restarts': N_RESTARTS,
    'test_run_fraction': TEST_RUN_FRACTION,
    'training_config': asdict(cfg_for_setup(SETUP_A)),  # same hyperparameters for both
    'python_version': platform.python_version(),
    'torch_version': torch.__version__,
    'numpy_version': np.__version__,
    'pandas_version': pd.__version__,
    'device': str(DEVICE),
    'use_amp': USE_AMP,
    'cuda_device_name': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
}
(RESULTS_DIR / 'metadata.json').write_text(json.dumps(md, indent=2))
print(json.dumps(md, indent=2))

## Cell 8: Run the sweep

Estimated wall time: 8 (site, setup) pairs × (G-NAVAR + Pairwise) × 3 restarts.
On T4 or A100, ~15-25 minutes total. Resume-safe.

In [ ]:
df = run_beijing_sweep()
print(f'\nDone. Total rows: {len(df)}')

## Cell 9: Cross-site consistency summary

In [ ]:
def write_beijing_summary(df):
    df = df.copy()
    lines = [
        'Beijing Air Quality: G-NAVAR cross-site application',
        '=' * 70,
        f'Trials: {len(df)} (site, setup) pairs',
        f'Sites: {SITES}',
        f'Setups: {[s["name"] for s in SETUPS]}',
        '',
        'Per-(site, setup) summary:',
        '',
    ]
    for _, row in df.iterrows():
        lines.append(f"  {row['site']:<15s} {row['setup']:<22s}  "
                     f"r_eff={row['r_eff']:5.3f}  "
                     f"MSE Gn/Pw test: {row['mse_test_gnavar']:.4f}/{row['mse_test_pairwise']:.4f}  "
                     f"(ratio {row['mse_ratio_pw_to_gn']:.2f}x)  "
                     f"expected modulator found: {row['hypothesis_modulator_found']}")
    lines.append('')

    # Cross-site consistency: per setup, count sites where expected modulator was found
    lines.append('Cross-site consistency:')
    for setup in SETUPS:
        sub = df[df['setup'] == setup['name']]
        n_found = int(sub['hypothesis_modulator_found'].sum())
        n_total = len(sub)
        lines.append(f"  Setup {setup['name']} ({setup['expected_edge_source']} -> {setup['target']} "
                     f"modulated by {setup['expected_modulator']}):")
        lines.append(f"    Expected modulator recovered in {n_found}/{n_total} sites")
        # Which sites succeeded vs failed?
        succ_sites = sub[sub['hypothesis_modulator_found']]['site'].tolist()
        fail_sites = sub[~sub['hypothesis_modulator_found']]['site'].tolist()
        if succ_sites:
            lines.append(f"      Recovered:     {succ_sites}")
        if fail_sites:
            lines.append(f"      Not recovered: {fail_sites}")
        # Mean test-MSE ratio per setup
        mse_ratio_mean = sub['mse_ratio_pw_to_gn'].mean()
        lines.append(f"    Mean test-MSE ratio (Pw/Gn): {mse_ratio_mean:.3f}x")
        # Triv score for the expected edge
        lines.append(f"    Expected-edge triv score per site:")
        for _, row in sub.iterrows():
            lines.append(f"      {row['site']:<15s} score = {row['expected_edge_triv_score']:.4f}  "
                         f"(threshold = 0.001)")
        lines.append('')

    # Effective rank vs hypothesis recovery
    lines.append('Effective rank vs hypothesis recovery (diagnostic check):')
    lines.append(f'  Where r_eff > 3, expected modulator found: '
                 f'{int(df[(df["r_eff"] > 3) & df["hypothesis_modulator_found"]].shape[0])}/'
                 f'{int(df[df["r_eff"] > 3].shape[0])} cases')
    lines.append(f'  Where r_eff <= 3, expected modulator found: '
                 f'{int(df[(df["r_eff"] <= 3) & df["hypothesis_modulator_found"]].shape[0])}/'
                 f'{int(df[df["r_eff"] <= 3].shape[0])} cases')

    summary = '\n'.join(lines)
    (RESULTS_DIR / 'summary.txt').write_text(summary)
    print(summary)

write_beijing_summary(df)

## Cell 10: Full modulator-set table per fit

In [ ]:
# Expand modulator dicts to a long-form table for inspection
rows = []
for _, row in df.iterrows():
    mod_dict = json.loads(row['modulator_dict_json'])
    triv_scores = json.loads(row['triv_scores_json'])
    for source, modulators in mod_dict.items():
        rows.append({
            'site': row['site'],
            'setup': row['setup'],
            'target': row['target'],
            'source': source,
            'discovered_modulators': ','.join(modulators) if modulators else '(none)',
            'n_modulators': len(modulators),
        })
df_mods = pd.DataFrame(rows)
df_mods.to_csv(RESULTS_DIR / 'modulators.csv', index=False)
print(f'Wrote {RESULTS_DIR / "modulators.csv"}')
print(f'\nModulator structure across all (site, setup, source) triples:')
print(df_mods.to_string(index=False))

## Cell 11: Comparison visualization

In [ ]:
def plot_beijing_comparison(df):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Panel A: per-(site,setup) MSE comparison (paired bars)
    labels = [f"{row['site'][:6]}\n{row['setup'][0]}" for _, row in df.iterrows()]
    x = np.arange(len(df))
    w = 0.35
    axes[0].bar(x - w/2, df['mse_test_gnavar'], width=w, label='G-NAVAR', color='C0')
    axes[0].bar(x + w/2, df['mse_test_pairwise'], width=w, label='Pairwise', color='C3')
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(labels, fontsize=9, rotation=0)
    axes[0].set_ylabel('Held-out forecast MSE')
    axes[0].set_title('Forecast MSE per (site, setup)')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3, axis='y')

    # Panel B: r_eff per case, marked by hypothesis-found
    colors = ['C2' if found else 'C1' for found in df['hypothesis_modulator_found']]
    axes[1].bar(x, df['r_eff'], color=colors)
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(labels, fontsize=9, rotation=0)
    axes[1].set_ylabel(r'Effective rank $r_\mathrm{eff}$ (max = n_sources $\times$ K)')
    axes[1].set_title('Pre-fit diagnostic by case (green = expected modulator found)')
    axes[1].grid(True, alpha=0.3, axis='y')

    fig.suptitle('Beijing Air Quality: G-NAVAR cross-site application', y=1.02)
    fig.tight_layout()
    out = RESULTS_DIR / 'comparison.png'
    fig.savefig(out, dpi=130, bbox_inches='tight')
    plt.show()
    print(f'Saved {out}')

plot_beijing_comparison(df)

## Cell 12: Triv-score ranking analysis (the substantive result)

The binary "expected modulator found" check in Cell 9 is one-sided: it returns True
whenever the expected modulator's gate triviality score is above the threshold, which
happens for *every* candidate modulator in real-data fits. With L1 lambda=0.005 (tuned
for the synthetic DGP) and dense real-data interactions, no gates get pruned to triviality.

The real-data signal lives in the **ranking** of triv scores per edge. For each
(site, setup), we examine the edge from the expected source to the target, then rank
all candidate modulators by their gate triviality score. The expected modulator's rank
(#1 if it dominates, lower if not) tells us whether G-NAVAR identifies the hypothesized
mechanism as the principal modulator.

Outputs:
- `triv_score_rankings.csv` — per-(site, setup) ranking with margins
- `substantive_findings.txt` — narrative summary including MSE win counts and diagnostic check

In [ ]:
import json
import pandas as pd

# Load fresh from CSV in case the in-memory df has been modified upstream
df = pd.read_csv(RESULTS_CSV)

# Per-setup: expected source and the candidate modulators for that edge
setup_meta = {s['name']: {'expected_source': s['expected_edge_source'],
                          'expected_modulator': s['expected_modulator'],
                          'all_sources': s['sources']}
              for s in SETUPS}

ranking_rows = []
print('Triv-score rankings per (site, setup) for the expected edge:')
print('=' * 75)

for _, row in df.iterrows():
    site  = row['site']
    setup = row['setup']
    meta  = setup_meta[setup]
    exp_src = meta['expected_source']
    exp_mod = meta['expected_modulator']
    candidates = [s for s in meta['all_sources'] if s != exp_src]
    triv = json.loads(row['triv_scores_json'])

    # Triv scores for each candidate as a modulator of exp_src
    edge_scores = {}
    for cand in candidates:
        key = f'{exp_src}<-{cand}'
        if key in triv:
            edge_scores[cand] = triv[key]

    # Rank descending
    ranked = sorted(edge_scores.items(), key=lambda x: -x[1])
    rank_names = [name for name, _ in ranked]
    exp_rank = rank_names.index(exp_mod) + 1
    top_name, top_score = ranked[0]
    exp_score = edge_scores[exp_mod]
    second_score = ranked[1][1] if len(ranked) > 1 else 0.0
    top_margin = top_score / second_score if second_score > 1e-12 else float('inf')

    ranking_rows.append({
        'site': site, 'setup': setup,
        'expected_source': exp_src, 'expected_modulator': exp_mod,
        'expected_rank': exp_rank, 'expected_triv_score': exp_score,
        'top_modulator': top_name, 'top_triv_score': top_score,
        'top_margin_over_2nd': top_margin,
        'ranking_full': '; '.join(f'{n}={s:.4f}' for n, s in ranked),
    })

    print(f"\n  {site:<14s}  {setup}")
    print(f"    Expected edge: {exp_src} -> target, expected modulator: {exp_mod}")
    print(f"    Top modulator: {top_name} (score={top_score:.4f}, "
          f"{top_margin:.1f}x over 2nd)")
    print(f"    Expected ({exp_mod}) ranked #{exp_rank} of {len(ranked)} "
          f"with score {exp_score:.4f}")
    for i, (name, score) in enumerate(ranked, 1):
        marker = ' <-- EXPECTED' if name == exp_mod else ''
        print(f'      {i}. {name:<6s}  triv = {score:.4f}{marker}')

df_rank = pd.DataFrame(ranking_rows)
df_rank.to_csv(RESULTS_DIR / 'triv_score_rankings.csv', index=False)
print(f"\n\nSaved {RESULTS_DIR / 'triv_score_rankings.csv'}")

# === Substantive summary ===
summary_lines = [
    'Beijing Air Quality: substantive findings from triv-score rankings',
    '=' * 75,
    'NOTE: The binary "expected modulator found" check from summary.txt is',
    'one-sided. L1 with lambda=0.005 (tuned on synthetic) does not prune',
    'gates on real data; every gate is above the triviality threshold.',
    'The substantive question is whether the expected modulator ranks',
    'FIRST among candidates by gate triv score.',
    '',
]

for setup_name in df_rank['setup'].unique():
    sub = df_rank[df_rank['setup'] == setup_name]
    meta = setup_meta[setup_name]
    summary_lines.append(f'Setup: {setup_name}')
    summary_lines.append(f'  Expected: {meta["expected_source"]} -> target modulated by {meta["expected_modulator"]}')
    summary_lines.append('')
    n_rank1 = int((sub['expected_rank'] == 1).sum())
    summary_lines.append(f'  Expected modulator ranked #1 in {n_rank1}/{len(sub)} sites')
    summary_lines.append('  Per-site rankings:')
    summary_lines.append(f'    {"site":<14s}  {"exp_rank":>9s}  {"top_modulator":>14s}  {"top_score":>10s}  {"margin_over_2nd":>16s}')
    for _, r in sub.iterrows():
        margin = f'{r["top_margin_over_2nd"]:.1f}x' if r['top_margin_over_2nd'] != float('inf') else 'inf'
        summary_lines.append(f'    {r["site"]:<14s}  {r["expected_rank"]:>9d}  {r["top_modulator"]:>14s}  {r["top_triv_score"]:>10.4f}  {margin:>16s}')
    if n_rank1 < len(sub):
        top_counts = sub['top_modulator'].value_counts().to_dict()
        # Convert numpy types to plain ints for clean printing
        top_counts_clean = {k: int(v) for k, v in top_counts.items()}
        summary_lines.append(f'  Alternative top-modulator pattern: {top_counts_clean}')
    summary_lines.append('')

summary_lines.append('Forecast MSE comparison (test set, lower is better):')
summary_lines.append(f'  {"site":<14s}  {"setup":<22s}  {"G-NAVAR":>9s}  {"Pairwise":>9s}  {"ratio":>7s}')
mse_wins_gn = 0
for _, row in df.iterrows():
    if row['mse_test_gnavar'] < row['mse_test_pairwise']:
        mse_wins_gn += 1
    summary_lines.append(f'  {row["site"]:<14s}  {row["setup"]:<22s}  '
                          f'{row["mse_test_gnavar"]:>9.4f}  {row["mse_test_pairwise"]:>9.4f}  '
                          f'{row["mse_ratio_pw_to_gn"]:>6.2f}x')
summary_lines.append(f'  G-NAVAR wins on test MSE: {mse_wins_gn}/{len(df)} cases')

summary_lines.append('')
summary_lines.append('Effective rank vs ranking-1 recovery (one row per case):')
r_eff_map = df.set_index(['site', 'setup'])['r_eff'].to_dict()
for _, r in df_rank.iterrows():
    reff = r_eff_map.get((r['site'], r['setup']), float('nan'))
    summary_lines.append(f'  {r["site"]:<14s}  {r["setup"]:<22s}  r_eff={reff:.3f}  expected_rank={r["expected_rank"]}')

substantive_summary = '\n'.join(summary_lines)
(RESULTS_DIR / 'substantive_findings.txt').write_text(substantive_summary)
print('\n' + substantive_summary)
print(f"\nSaved {RESULTS_DIR / 'substantive_findings.txt'}")